<a href="https://colab.research.google.com/github/dddddfrwe-ship-it/my-bot-app/blob/main/%E0%B8%AA%E0%B8%B3%E0%B9%80%E0%B8%99%E0%B8%B2%E0%B8%82%E0%B8%AD%E0%B8%87_%E0%B8%AA%E0%B8%B3%E0%B9%80%E0%B8%99%E0%B8%B2%E0%B8%82%E0%B8%AD%E0%B8%87_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch

In [ ]:
import torch
import torch.nn as nn
import math

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, nhead=8, num_layers=4, max_len=512):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        self.dropout = nn.Dropout(0.1)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=1024,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output = nn.Linear(embed_dim, vocab_size)

    def forward(self, x, mask=None):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.dropout(self.embed(x) + self.pos_embed(pos))
        x = self.transformer(x, src_key_padding_mask=mask)
        return self.output(x)

# vocab เบื้องต้น
special_tokens = ["<PAD>", "<UNK>", "<START>", "<END>", "<SEP>"]
vocab = {t: i for i, t in enumerate(special_tokens)}
idx2word = {i: t for t, i in vocab.items()}

model = TinyTransformer(vocab_size=10000)
params = sum(p.numel() for p in model.parameters())
print(f"parameters: {params:,}")
print(f"ขนาดประมาณ: {params*4/1024/1024:.1f} MB")
print("architecture พร้อมแล้วครับ!")

parameters: 8,420,112
ขนาดประมาณ: 32.1 MB
architecture พร้อมแล้วครับ!


In [ ]:
model = TinyTransformer(
    vocab_size=10000,
    embed_dim=512,
    nhead=8,
    num_layers=6,
    max_len=512
)
params = sum(p.numel() for p in model.parameters())
print(f"parameters: {params:,}")
print(f"ขนาดประมาณ: {params*4/1024/1024:.1f} MB")
print("พร้อมแล้วครับ!")

parameters: 23,128,848
ขนาดประมาณ: 88.2 MB
พร้อมแล้วครับ!


In [ ]:
model = TinyTransformer(
    vocab_size=10000,
    embed_dim=768,
    nhead=12,
    num_layers=12,
    max_len=512
)
params = sum(p.numel() for p in model.parameters())
print(f"parameters: {params:,}")
print(f"ขนาดประมาณ: {params*4/1024/1024:.1f} MB")

# เช็ค GPU memory
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM ว่าง: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")
else:
    print("ไม่มี GPU ครับ ใช้ CPU")

parameters: 63,044,368
ขนาดประมาณ: 240.5 MB
ไม่มี GPU ครับ ใช้ CPU


In [ ]:
import torch
import torch.nn as nn
import math

class NongGPT(nn.Module):
    def __init__(self, vocab_size=10000, embed_dim=512, nhead=8, num_layers=6, max_len=512):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        self.dropout = nn.Dropout(0.1)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=2048,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output = nn.Linear(embed_dim, vocab_size)
        self.max_len = max_len

    def make_causal_mask(self, sz):
        mask = torch.triu(torch.ones(sz, sz), diagonal=1).bool()
        return mask

    def forward(self, x):
        seq_len = x.size(1)
        pos = torch.arange(seq_len, device=x.device).unsqueeze(0)
        causal_mask = self.make_causal_mask(seq_len).to(x.device)

        emb = self.dropout(self.embed(x) + self.pos_embed(pos))
        out = self.transformer(emb, emb, tgt_mask=causal_mask, memory_mask=causal_mask)
        return self.output(out)

model = NongGPT(vocab_size=10000, embed_dim=512, nhead=8, num_layers=6)
params = sum(p.numel() for p in model.parameters())
print(f"NongGPT parameters: {params:,}")
print(f"ขนาด: {params*4/1024/1024:.1f} MB")
print("Decoder-only พร้อมแล้วครับ!")

NongGPT parameters: 35,736,336
ขนาด: 136.3 MB
Decoder-only พร้อมแล้วครับ!


In [ ]:
model = NongGPT(
    vocab_size=10000,
    embed_dim=1024,
    nhead=16,
    num_layers=24,
    max_len=1024
)

# แก้ไขบรรทัดนี้ให้พิมพ์ต่อกันชิดซ้ายสุด
params = sum(p.numel() for p in model.parameters())

print(f"NongGPT parameters: {params:,}")
print(f"ขนาด: {params*4/1024/1024:.1f} MB")

if torch.cuda.is_available():
    model = model.cuda()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM ว่าง: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")
    print("โหลด GPU สำเร็จ!")
else:
    print("ใช้ CPU ครับ")

NongGPT parameters: 323,946,256
ขนาด: 1235.8 MB
GPU: Tesla T4
VRAM ว่าง: 13.2 GB
โหลด GPU สำเร็จ!
